# 02 — Production Preprocessing Pipeline (Day 2, Stage 1)

Runs the full **Stage-1 dataset-cleaning pipeline** end to end:

`Validation → Cleaning → ColumnRemoval → TargetBuilder → Missing → Duplicates → Datatype → Integrity`

**Outputs:** a leakage-free, targeted `data/processed/kickstarter_processed.csv`, the
serialized pipeline (`artifacts/pipelines/`), and JSON reports (`outputs/reports/`).

In [1]:
import sys, json
from pathlib import Path

# Make the repo root importable and use it for all data paths (notebook cwd = notebooks/).
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import pandas as pd
from src.preprocessing.pipeline import PreprocessingPipeline
from src.preprocessing.serialization import PipelineSerializer

RAW = REPO / "data/raw/ks-projects-201801.csv"
PROCESSED = REPO / "data/processed/kickstarter_processed.csv"
REPORTS = REPO / "outputs/reports"
REPORTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)

repo: /Users/Yash/AI-Ventutre-Intelligence-Platform


## 1. Load raw data

In [2]:
raw = pd.read_csv(RAW)
print("raw shape:", raw.shape)
raw.head(3)

raw shape: (378661, 15)


,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,usd_pledged_real,usd_goal_real
0,1000002330,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09,1000.0,2015-08-11 12:12:28,0.0,failed,0,GB,0.0,0.0,1533.95
1,1000003930,Greeting From Earth: ZGAC Arts Capsule For ET,Narrative Film,Film & Video,USD,2017-11-01,30000.0,2017-09-02 04:43:57,2421.0,failed,15,US,100.0,2421.0,30000.00
2,1000004038,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26,45000.0,2013-01-12 00:20:50,220.0,failed,3,US,220.0,220.0,45000.00


## 2. Run the Stage-1 pipeline

In [3]:
pipe = PreprocessingPipeline(mode="train")
processed = pipe.fit_transform(raw)
print("processed shape:", processed.shape)
processed.head(3)

2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [empty_dataset] PASS: Dataset has 378,661 rows.


2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [duplicate_columns] PASS: No duplicate column names.


2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [required_columns] PASS: All 15 required columns present.


2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [unexpected_columns] PASS: No unexpected columns.


2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [target_values] PASS: All 'state' values are within the valid set.


2026-07-24 18:51:53 | avip.preprocessing.pipeline      | INFO     | [id_uniqueness] PASS: Identifier 'ID' is unique.


... (12 outputs left out)

2026-07-24 18:51:54 | avip.preprocessing.pipeline      | INFO     | Integrity (train): dropped 211 row(s), quarantined to data/interim/quarantined_invalid_rows.csv; 331464 remain.


2026-07-24 18:51:54 | avip.preprocessing.pipeline      | INFO     | PreprocessingPipeline done: 378661 -> 331464 rows, 12 columns in 1.44s.


processed shape: (331464, 12)


,name,category,main_category,currency,deadline,goal,launched,state,country,usd_goal_real,name_clean,target
0,The Songs of Adelaide & Abullah,Poetry,Publishing,GBP,2015-10-09,1000.0,2015-08-11 12:12:28,failed,GB,1533.95,the songs of adelaide abullah,0
1,Greeting From Earth: ZGAC Arts Capsule For ET,Narrative Film,Film & Video,USD,2017-11-01,30000.0,2017-09-02 04:43:57,failed,US,30000.00,greeting from earth zgac arts capsule for et,0
2,Where is Hank?,Narrative Film,Film & Video,USD,2013-02-26,45000.0,2013-01-12 00:20:50,failed,US,45000.00,where is hank,0


## 3. Per-step report (rows removed + timing)

In [4]:
report = pipe.get_report()
steps = pd.DataFrame(report["steps"]).T[["rows_in", "rows_out", "rows_removed", "seconds"]]
steps

,rows_in,rows_out,rows_removed,seconds
column_cleaner,378661.0,378661.0,0.0,0.017
text_cleaner,378661.0,378661.0,0.0,0.676
category_normalizer,378661.0,378661.0,0.0,0.098
column_remover,378661.0,378661.0,0.0,0.014
target_builder,378661.0,331675.0,46986.0,0.051
missing,331675.0,331675.0,0.0,0.085
duplicates,331675.0,331675.0,0.0,0.199
datatype,331675.0,331675.0,0.0,0.144
integrity,331675,331464,211,0.137


## 4. Integrity report (V1–V8)

In [5]:
integ = report["steps"]["integrity"]["report"]
print("dropped:", integ["rows_dropped"], "| remaining:", integ["rows_remaining"])
print("per-rule violations:", integ["violations_per_rule"])
print("flagged & kept:", integ["rows_flagged_kept"])

dropped: 211 | remaining: 331464
per-rule violations: {'V1': 1, 'V4': 0, 'V5': 0, 'V6': 0, 'V7': 210, 'V2': 92, 'V3': 22, 'V8': 0}
flagged & kept: {'V2': 92, 'V3': 22, 'V8': 0}


## 5. Sanity checks: no leakage, binary target

In [6]:
leakage = ["id", "pledged", "backers", "usd_pledged", "usd_pledged_real"]
assert not any(c in processed.columns for c in leakage), "LEAKAGE COLUMN PRESENT!"
assert set(processed["target"].unique()) <= {0, 1}, "target not binary!"
print("no leakage columns  :", True)
print("target is binary    :", True)
print("target distribution :")
print(processed["target"].value_counts(normalize=True).round(3))

no leakage columns  : True
target is binary    : True
target distribution :
target
0    0.596
1    0.404
Name: proportion, dtype: float64


## 6. Save processed dataset

In [7]:
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
processed.to_csv(PROCESSED, index=False)
print("saved:", PROCESSED, "|", processed.shape)

saved: /Users/Yash/AI-Ventutre-Intelligence-Platform/data/processed/kickstarter_processed.csv | (331464, 12)


## 7. Serialize pipeline + schema + metadata

In [8]:
ser = PipelineSerializer(artifacts_dir=str(REPO / "artifacts/pipelines"))
paths = ser.save(pipe, processed, raw_path=str(RAW))
print(json.dumps(paths, indent=2))
print("\nschema features:", ser.load_schema()["features"])

2026-07-24 18:51:55 | avip.preprocessing.serialization | INFO     | Serialized pipeline -> /Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/pipelines/preprocessing_pipeline.joblib (+ schema, metadata).


{
  "pipeline": "/Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/pipelines/preprocessing_pipeline.joblib",
  "schema": "/Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/pipelines/feature_schema.json",
  "metadata": "/Users/Yash/AI-Ventutre-Intelligence-Platform/artifacts/pipelines/metadata.json"
}

schema features: ['name', 'category', 'main_category', 'currency', 'deadline', 'goal', 'launched', 'state', 'country', 'usd_goal_real', 'name_clean', 'target']


## 8. Persist JSON reports

In [9]:
(REPORTS / "validation_report.json").write_text(json.dumps(report["validation"], indent=2))
(REPORTS / "integrity_report.json").write_text(json.dumps(integ, indent=2))
(REPORTS / "pipeline_report.json").write_text(json.dumps(
    {k: v for k, v in report.items() if k != "validation"}, indent=2, default=str))
print("wrote validation_report.json, integrity_report.json, pipeline_report.json")

wrote validation_report.json, integrity_report.json, pipeline_report.json


## 9. Reproducibility: reload and re-run

In [10]:
pipe2 = ser.load()
processed2 = pipe2.transform(raw)
print("reload identical shape:", processed.shape == processed2.shape)

2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [empty_dataset] PASS: Dataset has 378,661 rows.


2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [duplicate_columns] PASS: No duplicate column names.


2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [required_columns] PASS: All 15 required columns present.


2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [unexpected_columns] PASS: No unexpected columns.


2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [target_values] PASS: All 'state' values are within the valid set.


2026-07-24 18:51:55 | avip.preprocessing.pipeline      | INFO     | [id_uniqueness] PASS: Identifier 'ID' is unique.


... (9 outputs left out)

2026-07-24 18:51:56 | avip.preprocessing.pipeline      | INFO     | Integrity report: {'total_rows': 331675, 'violations_per_rule': {'V1': 1, 'V4': 0, 'V5': 0, 'V6': 0, 'V7': 210, 'V2': 92, 'V3': 22, 'V8': 0}, 'rows_dropped': 211, 'rows_flagged_kept': {'V2': 92, 'V3': 22, 'V8': 0}, 'rows_remaining': 331464, 'timestamp': '2026-07-24T13:21:56.885081+00:00'}


2026-07-24 18:51:56 | avip.preprocessing.pipeline      | WARNING  | Quarantined 211 row(s) -> data/interim/quarantined_invalid_rows.csv


2026-07-24 18:51:56 | avip.preprocessing.pipeline      | INFO     | Integrity (train): dropped 211 row(s), quarantined to data/interim/quarantined_invalid_rows.csv; 331464 remain.


reload identical shape: True


## ✅ Day 2 Deliverables Produced
- `data/processed/kickstarter_processed.csv` — leakage-free, targeted
- `data/interim/quarantined_invalid_rows.csv` — dropped rows + reasons
- `artifacts/pipelines/{preprocessing_pipeline.joblib, feature_schema.json, metadata.json}`
- `outputs/reports/{validation,integrity,pipeline}_report.json`